# 1.11 參數往哪裡改？

# 第 1 章：從猜下一個字開始

前置：讀過入門指南的 Python 與 tensor 部分即可。這一章只用幾個字；不要先開完整 Transformer。每段 code 都可單獨執行。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：旋鈕往哪邊轉，猜錯的代價會下降？**

藍線是 (w−3)²；橘點到綠點只走一小步。

這張圖保留靜態標註；先核對箭頭、位置與每個數字的意思。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/gradient.svg")))

**先想一想：**兩個參數對 loss 有不同影響，會收到相同梯度嗎？

requires_grad 把參數當成要追蹤的變量。forward 建立運算關係，backward 把最後 scalar loss 的敏感度傳回每個 leaf 參數。

**把直覺寫成數學：**gradient 的 shape 和參數相同：每一格都對應自己的偏導數。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
w = torch.tensor([1.0, 2.0], requires_grad=True)
loss = (w[0] - 3) ** 2 + 0.5 * w[1] ** 2
loss.backward()
print("參數", w.detach(), "梯度", w.grad)
assert torch.allclose(w.grad, torch.tensor([-4.0, 2.0]))

**如何讀結果：**grad 是 sensitivity，不是新的參數，也不是機率。

**只改一件事：**只改第二項係數，觀察哪一格梯度改變。
